
# 04 — Joint recurrent sales forecasting

One global GRU emits all 16 horizons together. This notebook trains the package implementation, inspects temporal results and demonstrates saved preprocessing reuse. It defaults to a small synthetic check. Set `RETAIL_FORECAST_MODE=full` before setup for the three-fold Kaggle study on CUDA. Synthetic scores are not empirical forecasting results.

See the [execution guide](docs/05_recurrent_forecasting.md) for setup, recovery and evidence limits.

In [ ]:
# Immutable package setup; see docs/01_reproducible_setup.md for clean local mode.
import importlib.util
import os
import re
import runpy
import subprocess
import sys
import tempfile
from getpass import getpass
from pathlib import Path

# The published academic URL can be supplied through the environment.
REPO_URL = os.environ.get("RETAIL_FORECAST_REPO_URL", "").strip()
if not REPO_URL:
    if (Path.cwd() / "pyproject.toml").is_file():
        REPO_URL = str(Path.cwd())
    else:
        REPO_URL = input("URL Git del repositorio académico: ").strip()
if not REPO_URL:
    raise ValueError("Indica la URL del repositorio académico publicado.")
CODE_REVISION = "f34244586752e0d09a26c72dcf52783f69762521"
MODELS = False
if sys.version_info[:2] not in {(3, 10), (3, 11), (3, 12), (3, 13)}:
    raise RuntimeError("Use Python 3.10–3.13 before installing or acquiring data.")
if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise RuntimeError("CODE_REVISION must be a reviewed full commit SHA.")
IN_COLAB = (
    importlib.util.find_spec("google") is not None
    and importlib.util.find_spec("google.colab") is not None
)
REPO_ROOT = Path(
    os.environ.get(
        "RETAIL_FORECAST_CHECKOUT",
        str(Path("/content" if IN_COLAB else tempfile.gettempdir()) / f"retail-academic-{CODE_REVISION}"),
    )
).absolute()
if REPO_ROOT.is_symlink() or any(p.is_symlink() for p in REPO_ROOT.parents):
    raise RuntimeError("Checkout path must not contain symlinks; use a new directory.")


def checkout_git(*arguments):
    return subprocess.run(
        ["git", "-C", str(REPO_ROOT), *arguments],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()


if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(REPO_ROOT)], check=True)
    # A squash merge can leave the reviewed SHA reachable only by its PR ref.
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "fetch", "--no-tags", "origin", CODE_REVISION],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(REPO_ROOT), "checkout", "--detach", CODE_REVISION],
        check=True,
    )
# Validate before executing even the shared helper from this checkout.
try:
    valid = (
        checkout_git("rev-parse", "--show-toplevel") == str(REPO_ROOT)
        and checkout_git("remote", "get-url", "origin") == REPO_URL
        and checkout_git("rev-parse", "HEAD") == CODE_REVISION
        and not checkout_git("status", "--porcelain", "--untracked-files=all")
    )
except subprocess.CalledProcessError:
    valid = False
if not valid:
    raise RuntimeError(
        "Dirty, incomplete or mismatched checkout. Preserve it; use a new directory."
    )
helper = runpy.run_path(str(REPO_ROOT / "src/retail_forecast/notebook_runtime.py"))
runtime_evidence = helper["bootstrap"](
    REPO_ROOT, CODE_REVISION, REPO_URL, models=MODELS, colab=IN_COLAB, neural=True
)
print(
    "Notebook revision:",
    os.getenv("RETAIL_FORECAST_NOTEBOOK_REVISION", "unrecorded; retain immutable notebook URL"),
)
os.chdir(REPO_ROOT)

In [ ]:
import json

import numpy as np
import pandas as pd
from IPython.display import display

from retail_forecast.acquisition import acquire_competition_data
from retail_forecast.artifacts import read_completed_run
from retail_forecast.config import load_config
from retail_forecast.data import required_files_exist, validate_dataset, write_synthetic_dataset
from retail_forecast.eda import validate_kaggle_hashes
from retail_forecast.pipeline import predict, request
from retail_forecast.recurrent import recurrent_experiment
from retail_forecast.recurrent_feasibility import recurrent_feasibility

MODE = os.getenv("RETAIL_FORECAST_MODE", "smoke")
if MODE not in {"smoke", "full"}:
    raise ValueError("RETAIL_FORECAST_MODE must be smoke or full")
config = load_config(
    REPO_ROOT / ("configs/gru.yaml" if MODE == "full" else "configs/gru-smoke.yaml")
)
print("Mode:", MODE, "Policy:", config.data.availability_policy)
print("Recipe:", config.recurrent)
print("Runtime identity:", runtime_evidence)

## Data and availability

Existing competition CSVs or a local ZIP are reused. Full acquisition needs the owner's Kaggle access; only missing data invokes the hidden token prompt. As-of inputs exclude future sales/transactions, oil and source events. Historical promotions proxy plan vintages; calendar/store metadata also have assumed historical availability.

In [ ]:
if config.data.source == "synthetic":
    write_synthetic_dataset(config)
elif not required_files_exist(config.paths.raw_data_dir):
    acquire_competition_data(
        config.paths.raw_data_dir, (REPO_ROOT / "data", REPO_ROOT), token_reader=getpass
    )
if MODE == "full":
    validate_kaggle_hashes(config.paths.raw_data_dir)
display(pd.DataFrame([validate_dataset(config).to_dict()]))

## Fixed recipe and causal scaling

Each sample has history `[56, 8]`, future covariates `[16, 12]`, six static category codes and one joint 16-output target. Missing input dates retain masks; observed zero remains observed. A single GRU layer encodes history; embeddings and the planned future covariates enter a dense output head.

At outer origin O, internal cutoff C=O−16 selects epochs on C+1…O. Scalers fit only eligible training rows. Fresh preprocessing and weights are then refit through O for the selected count. The loss reverses target standardization before original-log MSE so equal original-log errors receive equal weight across series.

The resource probe uses internal training windows and computes no outer score. Its approximate maximum-duration estimate does not guarantee runtime availability. Any smaller recipe requires registration before outer scoring. Probe cost is additional to primary training.

In [ ]:
probe = recurrent_feasibility(config)
probe_summary = json.loads(Path(probe["summary"]).read_text())
display(
    pd.DataFrame(
        [
            {
                k: probe_summary[k]
                for k in (
                    "phase",
                    "outer_scores_computed",
                    "device_name",
                    "host_peak_bytes",
                    "wall_seconds",
                    "estimated_max_primary_seconds",
                )
            }
        ]
    )
)
print("Probe evidence:", probe["run_dir"])

In [ ]:
study = recurrent_experiment(config)
read_completed_run(Path(study["run_dir"]))
summary = json.loads(Path(study["summary"]).read_text())
metrics = pd.read_csv(Path(study["run_dir"]) / "metrics/recurrent_results.csv")
oof = pd.read_parquet(study["oof"])
assert len(oof) == summary["oof_rows"] == summary["expected_rows"]
assert oof["lineage_status"].eq("complete").all()
print("Study evidence:", study["run_dir"])
display(
    pd.DataFrame(
        [
            {
                k: summary[k]
                for k in (
                    "empirical_results_are_synthetic",
                    "oof_rows",
                    "rmsle",
                    "mae",
                    "signed_bias",
                    "wape",
                    "primary_training_seconds",
                )
            }
        ]
    )
)
display(metrics.loc[metrics["scope"].isin(["fold", "horizon"])])

## Inspect training behavior

The pooled OOF RMSLE is primary; mean fold RMSLE is a different descriptive statistic. Internal histories explain epoch selection. Outer errors do not choose settings. Inspect convergence and failures with accuracy and costs. No extra-seed diagnostic or model promotion is claimed here.

In [ ]:
selections = json.loads((Path(study["run_dir"]) / "metrics/recurrent_selection.json").read_text())
histories = []
for selected in selections:
    for stage in selected["stages"]:
        histories.extend(
            {"fold": selected["fold"], "stage": stage["stage"], **row} for row in stage["history"]
        )
display(pd.DataFrame(histories))
print("Pooled OOF RMSLE:", summary["rmsle"])
print("Mean fold RMSLE (descriptive):", metrics.loc[metrics["scope"].eq("fold"), "rmsle"].mean())
print("Resource evidence:", Path(study["run_dir"]) / "metrics/recurrent_costs.json")

## Reuse the learned scaler and weights

The next request reuses the first fold's persisted categories, centers/scales and weights. It does not fit a new scaler. This deliberately reproduces an existing origin, rather than adding an independent evaluation. Unknown series and wholly unobserved contexts fail; sparse supported contexts retain masks. Separate-process tests also exercise offline capsules after removing the original training run.

In [ ]:
bundle = f"{study['training_runs'][0]}:models/gru_global/fold_1"
normalized = request(config, origin=config.validation.origins[0])
reloaded = predict(config, bundle=bundle, request_dir=Path(normalized["request_dir"]))
forecast = pd.read_csv(reloaded["forecast"])
reference = oof.loc[
    oof["fold"].eq("fold_1"), ["store_nbr", "family", "target_date", "y_pred"]
].copy()
reference["target_date"] = pd.to_datetime(reference["target_date"]).dt.strftime("%Y-%m-%d")
matched = forecast.merge(
    reference, on=["store_nbr", "family", "target_date"], validate="one_to_one"
)
assert len(matched) == len(forecast) == len(reference)
np.testing.assert_allclose(matched["prediction"], matched["y_pred"], rtol=1e-5, atol=1e-4)
print("Frozen-state parity rows:", len(matched))
print("Prediction evidence:", reloaded["run_dir"])
display(forecast.head())

## Interpretation and limits

Explain the actual histories, scale reuse and error tables above. Synthetic mode verifies execution only. Full mode evaluates observed sales in three adjacent intervals; it does not establish annual reliability, calibrated intervals, unconstrained demand or inventory savings. Compare only matching as-of baseline keys and policy. The GRU need not beat LightGBM.

Clean local execution and clean Colab execution are separate evidence. Human interpretation review remains explicit. Interrupted or partial runs are not full results. Original report and earlier notebooks remain historical evidence.